# 공정한 전이(math + factual) distillation 과 **student 실패 분석** (Qwen3-32B → Qwen3-1.7B)

`agent_state_preact_probe_qwen3_1.7b_distilled.ipynb` 의 distillation 은 **math 전용**이라 공정한 전이
테스트가 아닙니다. 검색(factual) 행동은 teacher(32B)가 아니라 작은 모델이 만든 로그라 SFT 에 안 들어갑니다.

**왜 이렇게 됐나?** 원본 파이프라인에선 teacher 궤적 생성 스크립트 하나가 math+factual 을 **둘 다** 만듭니다:
```bash
bash scripts/inference/run_agent_teacher_train.sh     # README "Generate Trajectories from Teacher Agent"
```
그런데 *이 체크아웃엔* 32B **math 궤적만** 남아 있습니다(`logs/.../Qwen3-32B/math_*_train/`). factual 절반
(hotpotqa/musique train)의 32B 궤적은 디스크에 없습니다 → 그래서 기존 distillation 이 math 전용이 된 것.

**고치는 법은 단순합니다 — 기존 노트북과 *완전히 같은 방식*.** 위 스크립트로 factual 궤적까지 생성해두면,
`build_sft_dialogues` 에 넘기는 파일 목록에 그걸 더하기만 하면 됩니다:
```python
sft_dialogs = build_sft_dialogues(TEACHER_MATH_FILES + TEACHER_QA_FILES)   # ← factual 파일 추가가 전부
```

그 다음, 이렇게 *math+factual* 로 학습된 student 가 **어떤 경우에 / 왜** teacher 행동을 재현하지
못하는지 **teacher-forced step 비교**로 상세 분석합니다(조기 finalize · 검색 누락 · 긴 궤적에서 행동 전달 실패 등).

## Section 1 · Setup & Config

In [ ]:
import sys, os, json, ast, re, gc, warnings, logging
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "6")     # 1.7B 는 GPU 1장이면 충분
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

def _find_root() -> Path:
    p = Path(".").resolve()
    for _ in range(6):
        if (p / "logs" / "qa_results").exists():
            return p
        p = p.parent
    return Path("/workspace/Agent-Distillation")

ROOT = _find_root()
sys.path.append(str(ROOT / "exps_research"))          # train_utils 임포트용
print("Project root:", ROOT)

# ── teacher / student (기존 노트북과 동일) ───────────────────────────────────
TEACHER_NAME = "Qwen/Qwen3-32B"
STUDENT_NAME = "Qwen/Qwen3-1.7B"
DTYPE        = torch.bfloat16
DEVICE_MAP   = "auto"

# math+factual 로 distillation 한 *공정* student 체크포인트
STUDENT_CKPT = ROOT / "training_outputs" / "qwen3-1.7B" / "agent_distill_from_qwen3_32b_mathqa"

TARGET_STATES = ["retrieve", "symbolic_math", "numeric_compute", "finalize", "compute", "inspect"]

SEED              = 42
N_EVAL_PER_DOMAIN = 200      # teacher-forced 비교에 쓸 도메인별 step 수
GEN_MAX_NEW       = 512
GEN_BATCH         = 8

# distillation(SFT) 하이퍼파라미터 — 기존 노트북과 동일
DISTILL_EPOCHS, DISTILL_LR = 2, 2e-4
DISTILL_MAXLEN, DISTILL_BS, DISTILL_GA = 4096, 1, 8
MAX_TRAIN_DIALOGS = None      # None=전체. 빠른 시험시 예: 200
FORCE_RETRAIN     = False
RUN_TF_GEN        = True      # teacher-forced student 생성 (캐시 없을 때만)

DISTILL_DATA_DIR = ROOT / "data" / "distill"; DISTILL_DATA_DIR.mkdir(parents=True, exist_ok=True)
FAIL_DIR         = ROOT / "outputs" / "agent_distill_failure"; FAIL_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED); torch.manual_seed(SEED)
print("ckpt exists:", (STUDENT_CKPT / "adapter_config.json").exists(),
      "| cuda:", torch.cuda.is_available(), "| visible:", torch.cuda.device_count())

## Section 2 · 파싱 helper (기존 노트북과 동일)

기존 distilled 노트북의 `load_jsonl / get_messages / role_of / text_of / code_of / label_state`
를 그대로 재사용합니다. 추가로 act 에서 *도구/finalize* 를 뽑는 helper(`tools_of` 등)만 더합니다 —
실패 분석용입니다.


In [ ]:
def load_jsonl(f):
    rows = []
    for line in open(f):
        line = line.strip()
        if not line: continue
        try: d = json.loads(line)
        except Exception: continue
        if isinstance(d, str):
            try: d = ast.literal_eval(d)
            except Exception:
                try: d = json.loads(d)
                except Exception: continue
        if isinstance(d, dict): rows.append(d)
    return rows

def get_messages(d):
    ld = d.get("log_data")
    if isinstance(ld, str):
        try: ld = ast.literal_eval(ld)
        except Exception:
            try: ld = json.loads(ld)
            except Exception: return []
    return ld.get("messages", []) if isinstance(ld, dict) else []

def role_of(m): return str(m.get("role")).split(".")[-1].lower().replace("-", "_")
def text_of(m):
    c = m.get("content")
    if isinstance(c, list): return " ".join(x.get("text","") for x in c if isinstance(x,dict))
    return str(c)

def code_of(t):
    m = re.search(r"```py(.*?)```", t, re.S) or re.search(r"Code:\s*(.*)$", t, re.S)
    return m.group(1) if m else t

def label_state(code):
    c = code.lower()
    if re.search(r"\b(retriever|web_search|wiki|document_qa|image_search|visit_webpage)\b|\bsearch\(", c): return "retrieve"
    if re.search(r"sympy|symbols\(|\bsolve\(|\bEq\(|simplify|integrate|\bdiff\(|expand\(|factor\(", code): return "symbolic_math"
    if re.search(r"\bnp\.|numpy|\bmath\.|factorial|comb\(|sqrt\(|\.evalf", code): return "numeric_compute"
    flat = re.sub(r"\s", "", code)
    if re.match(r"^final_answer\(.*\)$", flat) or (re.search(r"final_answer\(", c) and not re.search(r"\bfor\b|\bwhile\b|\bdef\b|=", code)): return "finalize"
    if re.search(r"\bfor\b|\bwhile\b|\bdef\b|range\(|append\(", code): return "compute"
    if re.search(r"\bprint\(", c): return "inspect"
    return "other"

# ── 실패 분석용 추가 helper ──────────────────────────────────────────────────
def tools_of(code):
    c = code.lower(); t = set()
    if re.search(r"\b(retriever|web_search|wiki|document_qa|image_search|visit_webpage)\b|\bsearch\(", c): t.add("retrieve")
    if re.search(r"sympy|symbols\(|\bsolve\(|\bEq\(|simplify|integrate|\bdiff\(|expand\(|factor\(", code): t.add("sympy")
    if re.search(r"\bnp\.|numpy", code): t.add("numpy")
    if re.search(r"\bmath\.|factorial|comb\(|sqrt\(|\.evalf", code): t.add("math")
    return sorted(t)

def finalizes(code): return bool(re.search(r"final_answer\(", code))
def final_arg(act):
    m = re.search(r"final_answer\((.*)\)", act, re.S)
    return m.group(1).strip() if m else None

print("helper ready.")

## Section 3 · Distillation dataset 생성 (math **+ factual**, 정답필터)

기존 노트북 Section 3 과 **동일한 `build_sft_dialogues`** 를 쓰되, 입력 파일에 **factual(32B) 궤적**을 더하고
스크립트의 **`--do_filtering` 과 동일하게 정답필터**를 적용합니다(정답 궤적만 SFT 에 사용).

**스크립트(`run_agent_teacher_train.sh`)와의 대응:**

| 스크립트가 하는 일 | 본 노트북 |
|---|---|
| `DATASETS=(hotpotqa, math, math2)` | `TEACHER_QA_FILES + TEACHER_MATH_FILES` (동일 3개) |
| `run_experiment --experiment_type agent` 로 궤적 *생성* | 외부 산출물 jsonl 을 소비 (생성은 스크립트 몫) |
| **`--do_filtering`** → `filtered_data/*_filtered.jsonl` (정답만) | `filtered_or_raw()` 로 **filtered 우선 사용** (= 정답필터) |

> - **생성은 스크립트, 학습은 노트북**(Section 4) — 둘은 별개 스텝입니다.
> - `--do_filtering` 산출물(`filtered_data/`)이 있으면 그걸(정답만) 쓰고, 없으면 raw 로 폴백하며 경고합니다.
>   raw 는 정답+오답이 섞여 있어(예: math_1000 → 1000개 중 정답 554개) 정답필터가 안 된 상태이니,
>   `--do_filtering` 을 켠 채로 스크립트를 돌려 filtered 를 만들어 두는 것이 스크립트와 정확히 동일합니다.
> - **teacher 모델 일치:** 스크립트 기본 `BASE_MODEL="Qwen/Qwen2.5-32B-Instruct"` → 기존 math 궤적과 맞추려면
>   **`Qwen/Qwen3-32B`** 로 바꿔 실행. factual 산출 위치는 `logs/.../Qwen_Qwen3-32B/hotpotqa_1000_20250402_train/`.
> - **현재 이 체크아웃엔 32B math 궤적만** 있습니다(factual 없음, math_medium 은 filtered 도 없음).

In [ ]:
from datasets import Dataset

# 기존 노트북 Section 3 의 build_sft_dialogues 를 그대로 재사용
def build_sft_dialogues(files):
    dialogs = []
    for f in files:
        if not Path(f).exists():
            print("  [skip] not found:", Path(f).name); continue
        for d in load_jsonl(f):
            if "score" in d and not (d["score"] in (True, 1)):   # scored 파일이면 정답만(안전망)
                continue
            msgs = get_messages(d)
            if not msgs: continue
            conv = []
            for m in msgs:
                if not isinstance(m, dict): continue
                r = role_of(m)
                if r == "tool_call": continue                 # assistant code 와 중복
                role = "user" if r == "tool_response" else r
                conv.append({"role": role, "content": text_of(m)})
            if conv and conv[0]["role"] == "system" and any(c["role"] == "assistant" for c in conv):
                dialogs.append({"messages": conv})
    return dialogs

# ── math + factual teacher 궤적 (raw 경로 기준) — run_agent_teacher_train.sh 가 둘 다 생성 ──
TEACHER_MATH_FILES = [
    ROOT / "logs/qa_results/vllm/Qwen_Qwen3-32B/math_1000_20250414_train/Qwen3-32B_temp=0.0_seed=42_type=agent_steps=5.jsonl",
    ROOT / "logs/qa_results/vllm/Qwen_Qwen3-32B/math_medium_1000_20250430_train/Qwen3-32B_temp=0.0_seed=42_type=agent_steps=5.jsonl",
]
TEACHER_QA_FILES = [
    ROOT / "logs/qa_results/vllm/Qwen_Qwen3-32B/hotpotqa_1000_20250402_train/Qwen3-32B_temp=0.0_seed=42_type=agent_steps=5.jsonl",
    # musique train 준비되면 같은 패턴으로 추가
]

# ── 정답필터 적용: --do_filtering 산출물(filtered_data/*_filtered.jsonl) 우선, 없으면 raw 폴백 ──
def filtered_or_raw(raw_path):
    raw = Path(raw_path)
    filt = raw.parent / "filtered_data" / raw.name.replace(".jsonl", "_filtered.jsonl")
    return (str(filt), True) if filt.exists() else (str(raw), False)

def resolve(files):
    out = []
    for p in files:
        fp, ok = filtered_or_raw(p)
        if Path(fp).exists():
            print(("  ✅ filtered(정답만)" if ok else "  ⚠️ RAW(미필터: --do_filtering 재실행 권장)"), Path(fp).name)
        out.append(fp)
    return out

print("[SFT 입력 파일 — 스크립트의 --do_filtering 과 동일하게 정답필터 적용]")
TEACHER_MATH_SRC = resolve(TEACHER_MATH_FILES)
TEACHER_QA_SRC   = resolve(TEACHER_QA_FILES)

sft_dialogs = build_sft_dialogues(TEACHER_MATH_SRC + TEACHER_QA_SRC)
if MAX_TRAIN_DIALOGS:
    sft_dialogs = sft_dialogs[:MAX_TRAIN_DIALOGS]

SFT_JSONL = DISTILL_DATA_DIR / "qwen3_32b_teacher_agent_sft_mathqa.jsonl"
with open(SFT_JSONL, "w") as f:
    for r in sft_dialogs:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

train_dataset = Dataset.from_list(sft_dialogs)
n_asst = sum(sum(1 for m in d["messages"] if m["role"] == "assistant") for d in sft_dialogs)
cov = Counter(label_state(code_of(m["content"]))
              for d in sft_dialogs for m in d["messages"] if m["role"] == "assistant")
print("\ndistillation 대화 수:", len(train_dataset), "| assistant(act) 턴:", n_asst)
print("state 커버리지:", dict(cov.most_common()))
print(f">>> retrieve 데모: {cov.get('retrieve',0)}/{n_asst}  ({100*cov.get('retrieve',0)/max(n_asst,1):.1f}%)")
print("saved:", SFT_JSONL)

In [ ]:
# 커버리지 막대그래프 — factual 을 더해 retrieve 가 채워졌는지 확인
fig, ax = plt.subplots(figsize=(8, 4))
order = [s for s in TARGET_STATES if s in cov] + [s for s in cov if s not in TARGET_STATES]
vals = [cov[s] for s in order]
bars = ax.bar(order, vals, color=["seagreen" if s == "retrieve" else "steelblue" for s in order])
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v, str(v), ha="center", va="bottom", fontsize=9)
ax.set_title("math+factual SFT 의 agent-state 커버리지 (retrieve 포함)")
ax.set_ylabel("# act turns"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()
if cov.get("retrieve", 0) < 10:
    print("⚠️ retrieve 데모가 거의 없음 → factual teacher 궤적이 아직 없습니다.")
    print("   먼저 `bash scripts/inference/run_agent_teacher_train.sh` 로 factual 궤적을 생성하세요.")

## Section 4 · Qwen3-1.7B 에 distillation (TRL SFT + LoRA)

기존 노트북 Section 4 와 **동일한 학습 설정**(LoRA r=64/α=128 all-linear, completion-only multi-turn loss).
체크포인트가 있으면 건너뜁니다(`FORCE_RETRAIN=True` 로 강제).


In [ ]:
adapter_ready = (STUDENT_CKPT / "adapter_config.json").exists()
if adapter_ready and not FORCE_RETRAIN:
    print("이미 distilled checkpoint 존재 → 학습 건너뜀:", STUDENT_CKPT)
else:
    from transformers import AutoTokenizer
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig
    from train_utils.utils import DataCollatorForCompletionOnlyLMMultiTurn

    tok = AutoTokenizer.from_pretrained(
        STUDENT_NAME, pad_token="<|endoftext|>", padding_side="left", add_eos_token=True)
    peft_config = LoraConfig(
        r=64, lora_alpha=128, target_modules="all-linear",
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    collator = DataCollatorForCompletionOnlyLMMultiTurn(
        "<|im_start|>assistant", instruction_template="<|im_start|>user", tokenizer=tok)
    train_args = SFTConfig(
        per_device_train_batch_size=DISTILL_BS, gradient_accumulation_steps=DISTILL_GA,
        max_length=DISTILL_MAXLEN, bf16=True, num_train_epochs=DISTILL_EPOCHS,
        learning_rate=DISTILL_LR, logging_steps=10, save_strategy="no",
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none", output_dir=str(STUDENT_CKPT))
    trainer = SFTTrainer(STUDENT_NAME, args=train_args, peft_config=peft_config,
                         data_collator=collator, train_dataset=train_dataset)
    trainer.train()
    trainer.save_model(str(STUDENT_CKPT))
    print("saved distilled adapter →", STUDENT_CKPT)
    del trainer; gc.collect(); torch.cuda.empty_cache()

## Section 5 · Teacher-forced step 비교 (실패 분석 원재료)

teacher 궤적의 매 step 에서 **teacher 의 prefix(직전까지의 대화)** 를 student 에 그대로 먹이고
student 의 *다음 act* 를 받아 teacher act 와 step 단위로 비교합니다.

- teacher = 32B 궤적(math train + factual train)
- student = 본 노트북에서 학습한 **math+factual** student
- (선택) 기존 *math 전용* 노트북의 결과 캐시 `tf_gen_math200_qa200.json` 가 있으면
  **math-only baseline** 으로 같이 로드해 전이 효과를 비교

> 무거운 생성 단계입니다. 캐시가 있으면 자동 로드, `RUN_TF_GEN=True` 면 없을 때만 생성합니다.


In [ ]:
# teacher 궤적 → 평가용 step (prefix messages + teacher act)
# 정답필터된(정답만) teacher 궤적을 reference 로 사용 — "올바른 teacher 행동을 student 가 재현하는가"
def build_eval_steps(files, domain, max_steps):
    steps = []
    for f in files:
        if not Path(f).exists(): continue
        for d in load_jsonl(f):
            if "score" in d and not (d["score"] in (True, 1)): continue
            msgs = get_messages(d)
            if not msgs: continue
            q = d.get("question", ""); ta = str(d.get("true_answer", d.get("answer", "")))
            conv = []
            for m in msgs:
                if not isinstance(m, dict): continue
                r = role_of(m)
                if r == "tool_call": continue
                role = "user" if r == "tool_response" else r
                content = text_of(m)
                if role == "assistant":
                    if any(c["role"] in ("user", "assistant") for c in conv):
                        steps.append({"domain": domain, "question": q, "true_answer": ta,
                                      "step_idx": sum(1 for c in conv if c["role"] == "assistant"),
                                      "prefix_msgs": [dict(c) for c in conv], "teacher_act": content})
                    conv.append({"role": "assistant", "content": content})
                else:
                    conv.append({"role": role, "content": content})
    if len(steps) > max_steps:
        idx = np.random.default_rng(SEED).permutation(len(steps))[:max_steps]
        steps = [steps[i] for i in sorted(idx)]
    return steps

EVAL_STEPS = (build_eval_steps(TEACHER_MATH_SRC, "math", N_EVAL_PER_DOMAIN)
              + build_eval_steps(TEACHER_QA_SRC,   "qa",   N_EVAL_PER_DOMAIN))
print("eval steps:", Counter(s["domain"] for s in EVAL_STEPS), "| total:", len(EVAL_STEPS))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def parse_act(act):
    cd = code_of(act)
    return {"state": label_state(cd), "tools": tools_of(cd),
            "finalizes": finalizes(cd), "final_arg": final_arg(act),
            "has_code": ("```py" in act) or ("Code:" in act)}

@torch.no_grad()
def tf_generate(steps):
    tok = AutoTokenizer.from_pretrained(STUDENT_NAME, trust_remote_code=True, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        STUDENT_NAME, torch_dtype=DTYPE, device_map=DEVICE_MAP, trust_remote_code=True)
    if (STUDENT_CKPT / "adapter_config.json").exists():
        model = PeftModel.from_pretrained(model, str(STUDENT_CKPT)).merge_and_unload()
        print("  LoRA merged from", STUDENT_CKPT.name)
    else:
        print("  ⚠️ adapter 없음 → base", STUDENT_NAME)
    model.eval()
    out = []
    for i in range(0, len(steps), GEN_BATCH):
        chunk = steps[i:i+GEN_BATCH]
        prompts = [tok.apply_chat_template(s["prefix_msgs"], tokenize=False, add_generation_prompt=True)
                   for s in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True,
                  max_length=DISTILL_MAXLEN).to(model.device)
        gen = model.generate(**enc, max_new_tokens=GEN_MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        plens = enc["attention_mask"].sum(1).tolist()
        for s, txt, plen in zip(chunk, texts, plens):
            tp, sp = parse_act(s["teacher_act"]), parse_act(txt)
            out.append({"domain": s["domain"], "step_idx": s["step_idx"], "question": s["question"],
                        "true_answer": s["true_answer"], "teacher_act": s["teacher_act"], "student_act": txt,
                        "teacher_state": tp["state"], "student_state": sp["state"],
                        "teacher_tools": tp["tools"], "student_tools": sp["tools"],
                        "student_finalizes": sp["finalizes"], "student_final_arg": sp["final_arg"],
                        "student_has_code": sp["has_code"], "prefix_len": int(plen)})
        if (i // GEN_BATCH) % 4 == 0: print(f"  gen {min(i+GEN_BATCH, len(steps))}/{len(steps)}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

# 공정(math+factual) student 결과
CACHE = FAIL_DIR / "tf_mathqa.json"
if CACHE.exists():
    tf_fair = json.load(open(CACHE)); print("cache:", CACHE.name, "| steps:", len(tf_fair))
elif RUN_TF_GEN and (STUDENT_CKPT / "adapter_config.json").exists():
    tf_fair = tf_generate(EVAL_STEPS)
    json.dump(tf_fair, open(CACHE, "w"), ensure_ascii=False, indent=1)
    print("saved:", CACHE.name, "| steps:", len(tf_fair))
else:
    tf_fair = None; print("공정 student 결과 없음 (체크포인트/캐시 필요)")

# (선택) 기존 math 전용 노트북 결과를 baseline 으로 로드
LEGACY = FAIL_DIR / "tf_gen_math200_qa200.json"
tf_math_only = json.load(open(LEGACY)) if LEGACY.exists() else None

tf = {}
if tf_fair: tf["math+factual"] = tf_fair
if tf_math_only: tf["math-only(baseline)"] = tf_math_only
print("분석 대상 student:", list(tf))

## Section 6 · 실패 모드 분류 (step 단위)

각 step 을 우선순위대로 분류합니다.

1. `retrieve_miss` — teacher 가 검색(retrieve)하는데 student 는 검색 안 함 (**factual 핵심 실패**)
2. `premature_finalize` — teacher 는 더 진행하는데 student 가 조기 `final_answer`
3. `tool_omission` — teacher 가 도구(sympy/np/검색) 쓰는데 student 는 코드/도구 없음
4. `wrong_tool` — 다른 도구 계열 사용
5. `state_mismatch` — 그 외 state 불일치 / `match` — 동일


In [ ]:
def classify(s):
    ts, ss = s["teacher_state"], s["student_state"]
    tt, st = set(s.get("teacher_tools") or []), set(s.get("student_tools") or [])
    if ts == "retrieve" and "retrieve" not in st:           return "retrieve_miss"
    if ts != "finalize" and s.get("student_finalizes"):     return "premature_finalize"
    if tt and (not s.get("student_has_code") or not st):    return "tool_omission"
    if tt and st and not (tt & st):                         return "wrong_tool"
    if ts != ss:                                            return "state_mismatch"
    return "match"

CATS = ["match", "retrieve_miss", "premature_finalize", "tool_omission", "wrong_tool", "state_mismatch"]

def failure_table(data):
    by = {"math": Counter(), "qa": Counter()}
    for s in data: by[s["domain"]][classify(s)] += 1
    return by

for tag, data in tf.items():
    by = failure_table(data)
    print(f"\n===== student = {tag} =====")
    for dom in ["math", "qa"]:
        tot = sum(by[dom].values())
        if not tot: continue
        print(f"  [{dom}] n={tot} | match={by[dom]['match']/tot:.0%} | " +
              " ".join(f"{c}={by[dom][c]}" for c in CATS if c != 'match' and by[dom][c]))

In [ ]:
# 실패 모드 분포 (stacked bar)
fig, axes = plt.subplots(1, len(tf), figsize=(6.2*len(tf), 4.6), squeeze=False)
colors = dict(zip(CATS, ["#9e9e9e","#d32f2f","#f57c00","#fbc02d","#7b1fa2","#1976d2"]))
for ax, (tag, data) in zip(axes[0], tf.items()):
    by = failure_table(data)
    doms = [d for d in ["math","qa"] if sum(by[d].values())]
    bottom = np.zeros(len(doms))
    for c in CATS:
        vals = np.array([by[d][c]/max(sum(by[d].values()),1) for d in doms])
        ax.bar(doms, vals, bottom=bottom, label=c, color=colors[c]); bottom += vals
    ax.set_title(f"student = {tag}"); ax.set_ylim(0,1); ax.set_ylabel("step 비율")
axes[0][-1].legend(bbox_to_anchor=(1.02,1), loc="upper left", fontsize=8)
plt.suptitle("step 단위 실패 모드 분포 (teacher 행동 대비)", y=1.02)
plt.tight_layout(); plt.show()

## Section 7 · **왜 실패하는가 — "teacher 궤적이 너무 어려워서"** 가설

"teacher 궤적이 너무 길고 복잡해서 student 가 특정 action 을 전달받지 못한다"를 정량화합니다.
step 별 난이도 feature 와 실패(=match 아님)의 관계를 봅니다.

- `step_idx` (궤적 깊이) · `teacher_act_len` (모방할 행동 길이) · `teacher_tool_n` · `prefix_len`(있으면)


In [ ]:
def approx_len(s): return len(re.findall(r"\S+", s or ""))

def featurize(data):
    rows = []
    for s in data:
        rows.append({"domain": s["domain"], "step_idx": s["step_idx"],
                     "teacher_act_len": approx_len(s["teacher_act"]),
                     "teacher_tool_n": len(s.get("teacher_tools") or []),
                     "prefix_len": s.get("prefix_len") or approx_len(s.get("question","")),
                     "fail": int(classify(s) != "match"), "cat": classify(s)})
    return rows

FEATS = ["step_idx", "teacher_act_len", "teacher_tool_n", "prefix_len"]
for tag, data in tf.items():
    rows = featurize(data)
    X = np.array([[r[f] for f in FEATS] for r in rows], float)
    y = np.array([r["fail"] for r in rows])
    print(f"\n===== {tag} | n={len(y)} | 전체 실패율={y.mean():.0%} =====")
    for f in ["step_idx", "teacher_act_len"]:
        v = X[:, FEATS.index(f)]; qs = np.quantile(v, [0,.33,.66,1.0])
        print(f"  [{f}] 구간별 실패율:", end=" ")
        for lo, hi in zip(qs[:-1], qs[1:]):
            m = (v>=lo) & (v<=hi)
            if m.sum(): print(f"[{lo:.0f}-{hi:.0f}]={y[m].mean():.0%}(n{m.sum()})", end="  ")
        print()
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        clf = LogisticRegression(max_iter=1000).fit(StandardScaler().fit_transform(X), y)
        print("  logit 계수(+면 어려울수록 실패↑):", dict(zip(FEATS, clf.coef_[0].round(3))))
    except Exception as e:
        print("  (회귀 생략:", e, ")")

In [ ]:
# 난이도(teacher act 길이) vs 실패율 곡선
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for tag, data in tf.items():
    rows = featurize(data)
    v = np.array([r["teacher_act_len"] for r in rows], float)
    y = np.array([r["fail"] for r in rows])
    o = np.argsort(v); v, y = v[o], y[o]
    bins = np.array_split(np.arange(len(v)), 6)
    ax.plot([v[b].mean() for b in bins if len(b)], [y[b].mean() for b in bins if len(b)], "-o", label=tag)
ax.set_xlabel("teacher act 길이 (≈모방 난이도, words)"); ax.set_ylabel("step 실패율")
ax.set_title("teacher 행동이 길수록 student 실패율 ↑"); ax.set_ylim(0,1)
ax.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Section 8 · 정성 사례 — 실패가 *어떻게* 일어나는가

각 실패 모드의 대표 사례를 teacher↔student 나란히 출력합니다. 특히 **긴/깊은 teacher 궤적에서
student 가 조기 종료하거나 도구를 빠뜨리는** 사례를 우선 표시합니다.


In [ ]:
def show_cases(data, cat, k=2):
    rows = [s for s in data if classify(s) == cat]
    if not rows: print(f"  ({cat}) 사례 없음"); return
    rows.sort(key=lambda s: (s["step_idx"], len(s["teacher_act"])), reverse=True)   # 어려운 것 우선
    print(f"\n############ {cat}  (총 {len(rows)}건) ############")
    for s in rows[:k]:
        print(f"\n--- [{s['domain']}] step {s['step_idx']} | Q: {s['question'][:90]}")
        print(f"  teacher: state={s['teacher_state']} tools={s.get('teacher_tools')}"
              f"  → student: state={s['student_state']} tools={s.get('student_tools')}")
        print("  TEACHER:\n    " + s["teacher_act"][:300].replace("\n","\n    "))
        print("  STUDENT:\n    " + s["student_act"][:300].replace("\n","\n    "))

DATA = tf.get("math+factual") or (list(tf.values())[0] if tf else None)
if DATA:
    for cat in ["retrieve_miss", "premature_finalize", "tool_omission", "wrong_tool"]:
        show_cases(DATA, cat, k=2)
else:
    print("표시할 데이터가 없습니다 (Section 5 캐시/생성 필요).")

## Section 9 · 요약

- **공정화는 단순합니다:** 기존 노트북의 `build_sft_dialogues` 를 그대로 쓰되 teacher 파일에
  **factual(32B) 궤적**(`run_agent_teacher_train.sh` 산출)을 더하면 SFT 에 `retrieve` 가 포함됩니다.
- **주요 실패 모드:** factual 의 `retrieve_miss` / `premature_finalize` — teacher 가 검색해야 하는
  step 에서 student 가 검색 대신 *조기 종료/환각*. (baseline 캐시가 있으면 math-only 대비 개선 확인)
- **왜:** `tool_omission` / `state_mismatch` 는 **teacher act 가 길고(act_len↑) 궤적이 깊을수록(step_idx↑)
  증가**(Section 7) — "teacher 행동이 너무 복잡/길어 student 가 그 action 을 전달받지 못한다"는 가설 지지.
- **다음 단계:** factual train split 확대(hotpotqa+musique), 난이도 상위 궤적의 *step 단축/하위목표 분해*
  후 재증류 효과 측정, `pre_act/pre_code` probe(기존 노트북)와 실패 라벨 결합해 *실패 사전 예측*.


In [ ]:
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("done.")